In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# ============================================================
# 1. LOAD DATASET
# ============================================================
df = pd.read_csv("indexData.csv")
print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# ============================================================
# 2. FEATURE ENGINEERING & TARGET VARIABLE
# ============================================================
# Since indexData.csv contains stock market data (Open, High, Low, Close),
# we can create a binary target: 1 if Close > Open (Market Up), else 0 (Market Down)
df["MarketUp"] = (df["Close"] > df["Open"]).astype(int)

# Handle missing values if any exist in pricing columns
df = df.dropna(subset=["Open", "High", "Low", "Close", "Volume", "Index"])

y = df["MarketUp"]

# ============================================================
# 3. SELECT PREDICTOR VARIABLES
# ============================================================
features = [
    "Index",
    "Open",
    "High",
    "Low",
    "Volume"
]

X = df[features]

# ============================================================
# 4. CATEGORICAL & NUMERICAL VARIABLES
# ============================================================
categorical_features = ["Index"]
numerical_features = ["Open", "High", "Low", "Volume"]

# ============================================================
# 5. TRAIN-TEST SPLIT
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ============================================================
# 6. PREPROCESSING
# ============================================================
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)

# ============================================================
# 7. BINOMIAL LOGISTIC REGRESSION & TRAINING
# ============================================================
l1_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=1.0,
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

l1_model.fit(X_train, y_train)

# ============================================================
# 8. PREDICTIONS
# ============================================================
y_pred_l1 = l1_model.predict(X_test)
y_prob_l1 = l1_model.predict_proba(X_test)[:, 1]

# ============================================================
# 9. MODEL EVALUATION
# ============================================================
accuracy = accuracy_score(y_test, y_pred_l1)

print("\n============================================")
print("BINOMIAL LOGISTIC REGRESSION RESULTS")
print("============================================")
print("\nAccuracy:")
print(round(accuracy, 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_l1))
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_l1,
        target_names=["Market Down", "Market Up"]
    )
)
print("\nROC-AUC:")
print(round(roc_auc_score(y_test, y_prob_l1), 4))

Dataset Shape: (112457, 8)

Columns:
['Index', 'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
